# Distributed Debugging

Open this notebook in **JupyterLab**, select your normal **Python 3** kernel, and set **Processes: 2** before running. This demo pauses the same eight-layer MLP on every process so you can inspect each rank through JupyterLab's standard debugger.

> This notebook assumes your environment contains the following dependency (apart from the jupyter-distributed extension):
> - torch

Define a small MLP with a breakpoint halfway through its forward pass. The local variables at the breakpoint include the current `layer_index`, input `x`, and latest `activation`.

In [ ]:
import torch
from torch import nn


class MLP(nn.Module):
    def __init__(self, width=128):
        super().__init__()
        self.layers = nn.ModuleList(nn.Linear(width, width) for _ in range(8))

    def forward(self, x):
        for layer_index, layer in enumerate(self.layers):
            activation = torch.relu(layer(x))
            if layer_index == 3:
                breakpoint()
            x = activation
        return x


model = MLP()

Activate the debugger with the bug button in the notebook toolbar, then run the next cell. When execution pauses:

1. Open the debugger sidebar and use **Rank** to choose a process.
2. Inspect `layer_index`, `x`, and `activation` in **Variables**, switching ranks to compare their values.
3. Open the Command Palette (**Ctrl/Cmd+Shift+C**), run **Debugger: Evaluate Code**, enter `activation.max().item()` in the debug console, and press `Shift+Enter`.
4. Use **Continue** to resume every rank and finish the cell.

In [ ]:
inputs = torch.randn(4, 128)
outputs = model(inputs)
outputs.shape